# Day 2~3 — IB 어댑터 학습 + Webull 매핑 가이드

NautilusTrader Interactive Brokers 어댑터 정독 결과 정리.
Webull 어댑터 구현 시 직접 참고할 패턴/매핑/함정.

**관련 파일**:
- 코드 레퍼런스: `venv/Lib/site-packages/nautilus_trader/adapters/interactive_brokers/`
- 인터페이스 스캐폴드: `venv/Lib/site-packages/nautilus_trader/adapters/_template/`
- 우리 작업물: `webull_adapter/` (이 노트북에 정리한 가이드 기반으로 구현 예정)

In [ ]:
# 코드 위치 확인
import nautilus_trader
from pathlib import Path

pkg_root = Path(nautilus_trader.__file__).parent
ib_root = pkg_root / 'adapters' / 'interactive_brokers'
tpl_root = pkg_root / 'adapters' / '_template'

print('NautilusTrader:', nautilus_trader.__version__)
print('IB adapter:', ib_root)
for p in sorted(ib_root.glob('*.py')):
    print(f'  {p.name:30s} {p.stat().st_size:>6d} bytes')
print('\nTemplate:', tpl_root)
for p in sorted(tpl_root.glob('*.py')):
    print(f'  {p.name:30s} {p.stat().st_size:>6d} bytes')

## 1. 5개 핵심 파일 역할 요약

| 파일 | 책임 | 베이스 클래스 | 줄 수 (IB) |
|------|------|--------------|-----------|
| `config.py` | 설정 DTO + paper/live 분기 | `LiveDataClientConfig`, `LiveExecClientConfig` | ~200 |
| `providers.py` | 종목 메타 → `Instrument` 변환 | `InstrumentProvider` | ~200 |
| `data.py` | 시세 스트림/히스토리 | `LiveMarketDataClient` | ~400 |
| `execution.py` | 주문/계좌/포지션 | `LiveExecutionClient` | ~500 |
| `factories.py` | 컴포넌트 조립 + 캐싱 | `LiveDataClientFactory`, `LiveExecClientFactory` | ~200 |

## 2. 필수 구현 메서드 (`_template/`로 검증)

### `LiveMarketDataClient`
```
_connect, _disconnect                          # 필수
_request_quote_ticks, _request_trade_ticks     # 히스토리 1회 조회
_request_bars, _request_instrument(s)
_subscribe_quote_ticks, _subscribe_bars        # 실시간 스트림
_subscribe_trade_ticks, _subscribe_instrument
_subscribe_order_book_deltas                   # Webull 미지원 (skip)
_subscribe_mark_prices, _subscribe_funding_rates  # Webull 미지원 (skip)
_unsubscribe_*                                  # 대응 해제
```

### `LiveExecutionClient`
```
_connect, _disconnect                          # 필수
_submit_order, _cancel_order, _modify_order    # 필수
_cancel_all_orders                             # 필수
_submit_order_list                             # OCO 등 (Webull paper 거부 가능)
generate_account_state                         # 필수
generate_order_status_report(s)                # 미체결/완료 주문
generate_fill_reports                          # 체결 내역
generate_position_status_reports               # 보유 포지션
generate_order_filled / canceled / rejected /  # 이벤트 발행 헬퍼
  submitted / accepted / triggered / updated
```

## 3. 메시지/콜백 흐름

### 시세 구독 (Webull MQTT)
```
Strategy.subscribe_bars()
    → MessageBus → LiveMarketDataClient._subscribe_quote_ticks()
        → mqtt_client.subscribe(f"quote/{symbol}")
MQTT broker push (별도 스레드)
    → on_message(topic, payload)
        → JSON 파싱 → QuoteTick 생성
        → loop.call_soon_threadsafe(self._handle_data, qt)
    → MessageBus.publish(QuoteTick)
        → Strategy.on_quote_tick / on_bar
```

### 주문 라이프사이클 (Webull HTTP + gRPC)
```
Strategy.submit_order()
    → MessageBus → LiveExecutionClient._submit_order(SubmitOrder cmd)
        → http.place_order(payload)            ← 동기/비동기 HTTP
        → response.order_id 저장
        → self.generate_order_submitted(...)
(이후 Webull gRPC 이벤트 스트림이 비동기로 푸시)
    → _on_order_event(WORKING)   → generate_order_accepted
    → _on_order_event(FILLED)    → generate_order_filled (FillReport)
    → _on_order_event(CANCELLED) → generate_order_canceled
    → _on_order_event(REJECTED)  → generate_order_rejected
```

## 4. IB → Webull 핵심 매핑

| 항목 | Interactive Brokers | Webull |
|------|---------------------|--------|
| **게이트웨이** | 단일 TWS Gateway (TCP) + ibapi | HTTP + MQTT + gRPC 3개 프로토콜 |
| **시세 데이터** | `reqMktData` 콜백 | MQTT 토픽 `quote/{symbol}` (확인 필요) |
| **주문 상태** | TWS `orderStatus` 콜백 | gRPC `OrderEvent` 스트림 |
| **종목 메타** | `reqContractDetails` | HTTP `GET /quote/get?symbol=AAPL` |
| **계좌 정보** | `reqAccountUpdates` 지속 스트림 | HTTP `GET /account/...` (주기 폴링) |
| **인증** | TWS 로그인 | App Key/Secret + JWT (1시간 만료) |
| **종목 ID** | `IBContract(symbol, secType, exchange)` | 단순 ticker (`"AAPL"`) |
| **paper/live** | `trading_mode: "paper"\|"live"` | `WEBULL_PAPER` 환경변수 + base_url 분기 |

### 구조 영향
IB는 `InteractiveBrokersClient` 단일 객체가 모든 것을 처리.
Webull은 `WebullAPIManager` 같은 래퍼를 만들어 3개 클라이언트를 캡슐화하는 것이 깔끔함:
```
WebullAPIManager
├── HTTPClient   (webullsdkmdata + webullsdktrade)  # 종목/계좌/주문
├── MQTTClient   (paho-mqtt + webullsdkquotescore)   # 시세 스트림
└── gRPCClient   (webullsdktradeeventscore)          # 주문 이벤트
```

## 5. 가장 막힐 부분 5개 (해결 전략)

### ① asyncio + paho-mqtt + grpcio 통합
**문제**: paho-mqtt는 동기 (`loop_forever`). grpcio는 자체 asyncio 지원. 셋을 한 이벤트 루프에서 굴리려면 패턴 필요.

**해결**:
```python
from concurrent.futures import ThreadPoolExecutor

self._mqtt_executor = ThreadPoolExecutor(max_workers=1)
self._mqtt.loop_start()  # paho-mqtt 백그라운드 네트워크 루프
self._mqtt.on_message = self._on_mqtt_message

def _on_mqtt_message(self, client, userdata, msg):
    # paho 콜백은 별도 스레드 → asyncio 루프로 안전 전환
    qt = parse_quote(msg.payload)
    self._loop.call_soon_threadsafe(self._handle_data, qt)
```

### ② Cython 타입 변환 (Price/Quantity/Money)
Webull은 float/str 반환 → NautilusTrader는 `Price(double, int_precision)` 요구.
`webull_adapter/parsing/conversions.py` 모듈 별도 작성 권장:
```python
def to_price(value: float | str, precision: int = 2) -> Price:
    return Price(float(value), precision)
```

### ③ 시간대 정규화 (Webull 응답 ET vs UTC 혼재)
**확인 필요**: Webull API 문서에서 응답 시각의 timezone 명시 안 되어 있을 가능성.
→ 모든 응답을 UTC로 강제 정규화하는 헬퍼:
```python
def normalize_ts(value, from_tz="America/New_York") -> pd.Timestamp:
    ts = pd.to_datetime(value)
    if ts.tz is None:
        ts = ts.tz_localize(from_tz)
    return ts.tz_convert("UTC")
```

### ④ Webull 에러 코드 → `OrderRejectReason` 매핑
**확인 필요**: Webull HTTP/gRPC 에러 코드 전체 목록.
초기 매핑 (실 에러 발생 시 점진적 보강):
```python
WEBULL_ERROR_TO_REJECT_REASON = {
    9001: "INSUFFICIENT_FUNDS",
    9002: "INVALID_ORDER_QUANTITY",
    9003: "INVALID_PRICE",
}
```

### ⑤ Place Order rate limit (1초당 1건)
**문제**: App ID 단위 rate limit. 동시 여러 종목 진입 시 두 번째 주문이 거부될 수 있음.
**해결**: `_submit_order` 내부에서 `asyncio.Semaphore(1)` + 최소 인터벌 대기:
```python
async def _submit_order(self, command):
    async with self._submit_lock:
        await self._throttle_until_next_slot()
        # ... HTTP place order
```

## 6. 구현 순서 (Day 4~7)

1. **Day 4 — providers.py** (가장 단순, HTTP 1회 호출)
   - `WebullInstrumentProvider.load_async(instrument_id)`
   - 단일 종목 (예: AAPL) Equity 객체 반환까지

2. **Day 4~5 — data.py (read-only 부분)**
   - `_request_quote_ticks` (HTTP snapshot 1회)
   - `_request_bars` (HTTP history)
   - MQTT 스트리밍은 Day 7로 미룸

3. **Day 6 — execution.py (read-only 부분)**
   - `_connect` + `generate_account_state`
   - `generate_order_status_reports`, `generate_position_status_reports`
   - `_submit_order`는 일단 `NotImplementedError` (Day 7+ paper에서)

4. **Day 7 — config.py + factories.py + 통합**
   - 4개 *Config 클래스 정리
   - Factory 작성
   - MQTT 시세 스트림 통합 (`_subscribe_quote_ticks`)
   - FastAPI에서 잔고 표시까지 동작

5. **2주차 — _submit_order paper trading 검증**
   - gRPC 이벤트 스트림 통합
   - 첫 매수→매도 사이클 paper에서 1회 성공

## 7. 다음 작업 시 참고할 코드 위치

| 궁금한 것 | 어디 보면 되나 |
|-----------|--------------|
| 새 어댑터 인터페이스 | `_template/` (스캐폴드) + `interactive_brokers/` (실제) |
| `LiveMarketDataClient` 베이스 | `live/data_client.py` |
| `LiveExecutionClient` 베이스 | `live/execution_client.py` |
| `OrderStatusReport` 등 보고서 타입 | `execution/reports.py` |
| `SubmitOrder` 등 명령 메시지 | `execution/messages.py` |
| Equity/Bar/Price 모델 | `model/instruments/`, `model/data.py`, `model/objects.py` |
| 백테스트 엔진 사용 패턴 | `examples/backtest/` (디렉토리), 우리 `backtests/run_sma_cross.py` |